# KazFin-Surface: прогон моделей

Одни и те же финансовые задачи в шести регистрах: английский, русский, нормативный казахский,
казахский на русской раскладке, казахско-русское смешение и смешение на раскладке.
Ноутбук прогоняет три модели, дополнительно проверяет казахскую часть KazMMLU и собирает
всё в `results.zip`.

**Перед запуском (справа, панель Settings):**
1. *Accelerator* → **GPU T4 x2**.
2. *Internet* → **On** (нужна подтверждённая по телефону учётная запись Kaggle).
3. *Add-ons → Secrets* → секрет **HF_TOKEN** со своим токеном Hugging Face, галочка «attached».

**Как запускать:**
1. Сначала **Run All** с `QUICK_TEST = True` (около 10–15 минут): проверка доступов и кода.
2. Затем `QUICK_TEST = False` → **Save Version → Save & Run All (Commit)**. Полный прогон идёт
   в фоне (ориентировочно 3–5 часов), браузер можно закрыть.
3. Готовый `results.zip` — во вкладке **Output** сохранённой версии.

In [ ]:
# ---- настройки ----
MODELS = [
    "inceptionai/Llama-3.1-Sherkala-8B-Chat",   # казахская модель MBZUAI / Inception
    "issai/LLama-3.1-KazLLM-1.0-8B",            # казахская модель ISSAI
    "Qwen/Qwen2.5-7B-Instruct",                 # многоязычная базовая линия
]
QUICK_TEST = True        # True: 2 задачи на шаблон, 10 вопросов KazMMLU на предмет
PER_TEMPLATE = 20        # задач на шаблон в полном прогоне (12 шаблонов x 6 регистров)
RUN_KAZMMLU = True
KAZMMLU_LIMIT = 100      # вопросов на предмет в полном прогоне
MAX_NEW_TOKENS = 768
BATCH_SIZE = 16

In [ ]:
import subprocess, sys, importlib, os, time
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi не найден")
import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPUs:", torch.cuda.device_count())
from packaging.version import Version
if Version(transformers.__version__) < Version("4.43"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.45", "accelerate"])
    print("transformers обновлён — перезапустите ядро (Run → Restart) и запустите заново")
assert torch.cuda.device_count() >= 1, "GPU не включён: Settings → Accelerator → GPU T4 x2"

In [ ]:
# ---- доступ к Hugging Face ----
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(os.environ["HF_TOKEN"])
problems = []
for m in MODELS:
    try:
        hf_hub_download(m, "config.json")
        print("ok  ", m)
    except Exception as e:
        problems.append(m); print("НЕТ ДОСТУПА:", m, "→", type(e).__name__)
if RUN_KAZMMLU:
    try:
        from huggingface_hub import list_repo_files
        all_files = list_repo_files("MBZUAI/KazMMLU", repo_type="dataset")
        kz = [x for x in all_files if "in kaz)" in x and x.endswith("test.csv")]
        if not kz:
            print("В KazMMLU не найдены папки '... in kaz)'. Папки в репозитории:",
                  sorted({x.split("/")[0] for x in all_files})[:40])
            raise SystemExit("Пришлите этот список в чат — поправлю отбор предметов. Или RUN_KAZMMLU = False.")
        hf_hub_download("MBZUAI/KazMMLU", kz[0], repo_type="dataset"); print("ok   MBZUAI/KazMMLU,", len(kz), "предметов на казахском")
    except SystemExit:
        raise
    except Exception as e:
        problems.append("MBZUAI/KazMMLU"); print("НЕТ ДОСТУПА: MBZUAI/KazMMLU →", type(e).__name__)
if problems:
    raise SystemExit("Откройте страницы на huggingface.co, примите условия доступа и запустите заново: "
                     + ", ".join(problems))

## Код эксперимента
Эти ячейки записывают модули рядом с ноутбуком. Править их здесь не нужно.

In [ ]:
%%writefile kk_surface.py
"""Surface-form variants of Kazakh text.

keyboard_flatten() reproduces how people type Kazakh on a Russian keyboard:
the nine Kazakh-specific Cyrillic letters are replaced by their nearest
Russian letters. The mapping is deterministic, so the meaning and the gold
answer of a task do not change - only the spelling does.

Code-switched variants are NOT generated here: naturalistic Kazakh-Russian
mixing (Russian stems with Kazakh suffixes, Russian terms inside Kazakh
syntax) is authored by a native speaker in the templates themselves.
"""
from __future__ import annotations

import random
import re

# Russian-keyboard substitutions (lower case). Upper case is derived below.
KEYBOARD_RU = {
    "ә": "а",
    "ғ": "г",
    "қ": "к",
    "ң": "н",
    "ө": "о",
    "ұ": "у",
    "ү": "у",
    "һ": "х",
    "і": "и",
}

# Common alternative: і is typed as Latin "i" (it sits on many phone layouts).
KEYBOARD_RU_LATIN_I = dict(KEYBOARD_RU, **{"і": "i"})

KAZAKH_SPECIFIC = set(KEYBOARD_RU) | {c.upper() for c in KEYBOARD_RU}


def _with_upper(mapping: dict[str, str]) -> dict[str, str]:
    full = dict(mapping)
    for src, dst in mapping.items():
        full[src.upper()] = dst.upper()
    return full


def keyboard_flatten(text: str, latin_i: bool = False, rate: float = 1.0,
                     seed: int | None = None) -> str:
    """Replace Kazakh-specific letters with Russian-keyboard equivalents.

    rate < 1.0 flattens only that share of eligible characters (people are
    often inconsistent within one message); rate = 1.0 flattens all of them.
    """
    mapping = _with_upper(KEYBOARD_RU_LATIN_I if latin_i else KEYBOARD_RU)
    if rate >= 1.0:
        return text.translate(str.maketrans(mapping))
    rng = random.Random(seed)
    out = []
    for ch in text:
        if ch in mapping and rng.random() < rate:
            out.append(mapping[ch])
        else:
            out.append(ch)
    return "".join(out)


def kazakh_letter_share(text: str) -> float:
    """Share of Cyrillic letters that are Kazakh-specific (0 for flattened text)."""
    cyr = [c for c in text if re.match(r"[Ѐ-ӿ]", c)]
    if not cyr:
        return 0.0
    return sum(c in KAZAKH_SPECIFIC for c in cyr) / len(cyr)


if __name__ == "__main__":
    sample = "Ағымдағы өтімділік коэффициентін есептеңіз. Несие сомасы қанша?"
    print(sample)
    print(keyboard_flatten(sample))
    print(keyboard_flatten(sample, latin_i=True))
    print(keyboard_flatten(sample, rate=0.5, seed=1))

In [ ]:
%%writefile templates_kk.py
"""Parallel financial-reasoning templates in EN / RU / KK, after FinChain.

Adapted from the FinChain template design (github.com/mbzuai-nlp/finchain,
Apache-2.0). Each template returns the same numbers and the same gold answer
in every language register, so any accuracy difference between registers is
caused by the surface form of the question, not by the task.

Registers produced per instance:
  en      - English (FinChain style)
  ru      - Russian
  kk      - standard written Kazakh
  kk_kb   - kk typed on a Russian keyboard      (derived automatically)
  kk_cs   - Kazakh-Russian code-switched, as people write in chats
            (authored by hand - MUST be reviewed by a native speaker)
  kk_cs_kb- kk_cs typed on a Russian keyboard   (derived automatically)

DRAFT WORDING: the Kazakh and code-switched sentences are first drafts and
need a native-speaker pass before any results are reported.
"""
from __future__ import annotations

import random

from kk_surface import keyboard_flatten

COMPANIES = ["Алатау Құрылыс", "Есіл Агро", "Сарыарқа Логистик",
             "Жетісу Трейд", "Ертіс Энерго"]
COMPANIES_EN = {"Алатау Құрылыс": "Alatau Qurylys", "Есіл Агро": "Esil Agro",
                "Сарыарқа Логистик": "Saryarqa Logistic", "Жетісу Трейд": "Zhetisu Trade",
                "Ертіс Энерго": "Ertis Energo"}


# ---------- number formatting -------------------------------------------- #
def money(x: float, lang: str) -> str:
    """1250000.5 -> '1,250,000.50' (en) or '1 250 000,50' (ru/kk)."""
    s = f"{x:,.2f}"
    if lang == "en":
        return s
    return s.replace(",", " ").replace(".", ",")


def pct(x: float, lang: str) -> str:
    s = f"{x:.2f}"
    return s if lang == "en" else s.replace(".", ",")


def _finish(item: dict) -> dict:
    q = item["question"]
    q["kk_kb"] = keyboard_flatten(q["kk"])
    q["kk_cs_kb"] = keyboard_flatten(q["kk_cs"])
    return item


# ---------- T1: simple interest, single repayment ------------------------ #
def t_simple_interest(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    p = float(rng.randint(5, 200) * 1_000_000)          # tenge
    r = round(rng.uniform(8.0, 24.0), 2)                  # % per year
    t = rng.randint(2, 7)                                 # years
    total = p * (1 + r / 100 * t)
    q = {
        "en": (f"{COMPANIES_EN[co]} LLP takes a loan of KZT {money(p,'en')} at a simple annual "
               f"interest rate of {pct(r,'en')}% for {t} years. Calculate the total "
               f"amount repaid at the end of the term."),
        "ru": (f"ТОО «{co}» получило кредит в размере {money(p,'ru')} тенге на {t} лет "
               f"под {pct(r,'ru')}% годовых (простые проценты). Рассчитайте общую сумму "
               f"к возврату в конце срока."),
        "kk": (f"«{co}» ЖШС {money(p,'kk')} теңге көлемінде {t} жылға жылдық "
               f"{pct(r,'kk')}% жай пайыз мөлшерлемесімен несие алды. Мерзім соңында "
               f"қайтарылатын жалпы соманы есептеңіз."),
        "kk_cs": (f"«{co}» ТОО {money(p,'kk')} теңге кредит алды, {t} жылға, ставкасы "
                  f"жылына {pct(r,'kk')}%, простой процентпен. Срок соңында барлығы "
                  f"қанша қайтарады, посчитайте."),
    }
    steps = [f"r = {r}/100 = {r/100:.4f}",
             f"Total = P × (1 + r × t) = {p:.2f} × (1 + {r/100:.4f} × {t}) = {total:.2f}"]
    return _finish({"template": "simple_interest", "level": "basic",
                    "question": q, "steps": steps, "answer": round(total, 2)})


# ---------- T2: annuity loan, monthly payment ---------------------------- #
def t_annuity_payment(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    p = float(rng.randint(10, 300) * 1_000_000)
    r = round(rng.uniform(10.0, 26.0), 2)
    years = rng.randint(1, 5)
    n = years * 12
    i = r / 100 / 12
    pay = p * i / (1 - (1 + i) ** (-n))
    q = {
        "en": (f"{COMPANIES_EN[co]} LLP takes an annuity loan of KZT {money(p,'en')} for {years} years "
               f"at {pct(r,'en')}% per year, repaid in equal monthly payments. Calculate "
               f"the monthly payment."),
        "ru": (f"ТОО «{co}» взяло аннуитетный кредит на {money(p,'ru')} тенге сроком "
               f"{years} лет под {pct(r,'ru')}% годовых с ежемесячными равными платежами. "
               f"Рассчитайте сумму ежемесячного платежа."),
        "kk": (f"«{co}» ЖШС {money(p,'kk')} теңгеге {years} жылға жылдық {pct(r,'kk')}% "
               f"мөлшерлемемен аннуитеттік несие алды. Төлемдер ай сайын тең сомамен "
               f"жүргізіледі. Ай сайынғы төлем сомасын есептеңіз."),
        "kk_cs": (f"«{co}» ТОО {money(p,'kk')} теңгеге {years} жылға аннуитетный кредит "
                  f"алды, ставка жылына {pct(r,'kk')}%. Платеж ай сайын бірдей. "
                  f"Ежемесячный платеж қанша болады?"),
    }
    steps = [f"i = {r}/100/12 = {i:.6f}", f"n = {years} × 12 = {n}",
             f"A = P × i / (1 − (1 + i)^−n) = {pay:.2f}"]
    return _finish({"template": "annuity_payment", "level": "intermediate",
                    "question": q, "steps": steps, "answer": round(pay, 2)})


# ---------- T3: current liquidity ratio from balance-sheet items --------- #
def t_current_ratio(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    cash = float(rng.randint(20, 400) * 1_000_000)
    ar = float(rng.randint(50, 900) * 1_000_000)
    inv = float(rng.randint(30, 700) * 1_000_000)
    cl = float(rng.randint(80, 1200) * 1_000_000)
    ca = cash + ar + inv
    ratio = ca / cl
    q = {
        "en": (f"Year-end data of {COMPANIES_EN[co]} LLP: cash KZT {money(cash,'en')}, short-term "
               f"receivables KZT {money(ar,'en')}, inventories KZT {money(inv,'en')}, "
               f"short-term liabilities KZT {money(cl,'en')}. Calculate the current "
               f"ratio to two decimal places."),
        "ru": (f"Данные ТОО «{co}» на конец года: денежные средства — {money(cash,'ru')} "
               f"тенге, краткосрочная дебиторская задолженность — {money(ar,'ru')} тенге, "
               f"запасы — {money(inv,'ru')} тенге, краткосрочные обязательства — "
               f"{money(cl,'ru')} тенге. Рассчитайте коэффициент текущей ликвидности "
               f"с точностью до двух знаков."),
        "kk": (f"«{co}» ЖШС-нің жыл соңындағы деректері: ақша қаражаты — "
               f"{money(cash,'kk')} теңге, қысқа мерзімді дебиторлық берешек — "
               f"{money(ar,'kk')} теңге, қорлар — {money(inv,'kk')} теңге, қысқа "
               f"мерзімді міндеттемелер — {money(cl,'kk')} теңге. Ағымдағы өтімділік "
               f"коэффициентін екі ондық белгіге дейін есептеңіз."),
        "kk_cs": (f"«{co}» ТОО-ның год соңындағы данныйлары: деньги {money(cash,'kk')} "
                  f"теңге, дебиторка {money(ar,'kk')} теңге, запастар {money(inv,'kk')} "
                  f"теңге, краткосрочные обязательства {money(cl,'kk')} теңге. Текущая "
                  f"ликвидность коэффициентін есептеңізші, два знака после запятой."),
    }
    steps = [f"Current assets = {cash:.0f} + {ar:.0f} + {inv:.0f} = {ca:.0f}",
             f"Current ratio = {ca:.0f} / {cl:.0f} = {ratio:.4f} ≈ {ratio:.2f}"]
    return _finish({"template": "current_ratio", "level": "basic",
                    "question": q, "steps": steps, "answer": round(ratio, 2)})


PEOPLE = ["Айгерім", "Ерлан", "Дана", "Нұрлан", "Мадина"]
PEOPLE_EN = {"Айгерім": "Aigerim", "Ерлан": "Erlan", "Дана": "Dana",
             "Нұрлан": "Nurlan", "Мадина": "Madina"}
MONTHS_EN = ["January", "February", "March", "April", "May", "June", "July",
             "August", "September", "October", "November", "December"]


def date_str(d, lang: str) -> str:
    """15 March 2026 (en) or 15.03.2026 (ru/kk)."""
    return f"{d.day} {MONTHS_EN[d.month - 1]} {d.year}" if lang == "en" else d.strftime("%d.%m.%Y")


def _annuity(p: float, r: float, n: int) -> float:
    i = r / 100 / 12
    return p * i / (1 - (1 + i) ** (-n))


# ---------- T4: bank deposit with monthly capitalisation ----------------- #
def t_deposit_compound(rng: random.Random) -> dict:
    who = rng.choice(PEOPLE)
    p = float(rng.randint(1, 50) * 100_000)
    r = round(rng.uniform(10.0, 17.0), 2)
    m = rng.choice([6, 9, 12, 18, 24, 36])
    fv = p * (1 + r / 100 / 12) ** m
    q = {
        "en": (f"{PEOPLE_EN[who]} places KZT {money(p,'en')} on a bank deposit at {pct(r,'en')}% "
               f"per year with monthly capitalisation of interest for {m} months. Calculate "
               f"the deposit balance at the end of the term."),
        "ru": (f"{who} вносит на банковский депозит {money(p,'ru')} тенге под {pct(r,'ru')}% "
               f"годовых с ежемесячной капитализацией процентов на {m} месяцев. Рассчитайте "
               f"сумму на депозите в конце срока."),
        "kk": (f"{who} банк салымына {money(p,'kk')} теңге салды: мөлшерлеме жылдық "
               f"{pct(r,'kk')}%, пайыздар ай сайын капиталданады, мерзімі {m} ай. Мерзім "
               f"соңындағы салым сомасын есептеңіз."),
        "kk_cs": (f"{who} депозитке {money(p,'kk')} теңге салды, годовой {pct(r,'kk')}%, "
                  f"капитализация ай сайын, срок {m} ай. Срок біткенде депозитте қанша "
                  f"болады?"),
    }
    steps = [f"i = {r}/100/12 = {r/1200:.6f}",
             f"FV = P × (1 + i)^m = {p:.2f} × (1 + {r/1200:.6f})^{m} = {fv:.2f}"]
    return _finish({"template": "deposit_compound", "level": "basic",
                    "question": q, "steps": steps, "answer": round(fv, 2)})


# ---------- T5: total interest (overpayment) on an annuity loan ---------- #
def t_loan_overpayment(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    p = float(rng.randint(10, 300) * 1_000_000)
    r = round(rng.uniform(10.0, 26.0), 2)
    years = rng.randint(1, 7)
    n = years * 12
    a = _annuity(p, r, n)
    over = a * n - p
    q = {
        "en": (f"{COMPANIES_EN[co]} LLP takes an annuity loan of KZT {money(p,'en')} for {years} "
               f"years at {pct(r,'en')}% per year, repaid in equal monthly payments. Calculate "
               f"the total interest paid over the life of the loan."),
        "ru": (f"ТОО «{co}» взяло аннуитетный кредит на {money(p,'ru')} тенге сроком {years} "
               f"лет под {pct(r,'ru')}% годовых с ежемесячными равными платежами. Рассчитайте "
               f"переплату по кредиту, то есть общую сумму процентов за весь срок."),
        "kk": (f"«{co}» ЖШС {money(p,'kk')} теңгеге {years} жылға жылдық {pct(r,'kk')}% "
               f"мөлшерлемемен аннуитеттік несие алды, төлемдер ай сайын тең сомамен "
               f"жүргізіледі. Несиенің бүкіл мерзімінде төленетін пайыздардың жалпы сомасын "
               f"(артық төлемді) есептеңіз."),
        "kk_cs": (f"«{co}» ТОО {money(p,'kk')} теңгеге {years} жылға аннуитетный кредит алды, "
                  f"ставка жылына {pct(r,'kk')}%, платеж ай сайын бірдей. Весь срок бойы "
                  f"переплата қанша болады, яғни процентке барлығы қанша төлейді?"),
    }
    steps = [f"A = P × i / (1 − (1 + i)^−n) = {a:.2f}, n = {n}",
             f"Interest = A × n − P = {a:.2f} × {n} − {p:.2f} = {over:.2f}"]
    return _finish({"template": "loan_overpayment", "level": "intermediate",
                    "question": q, "steps": steps, "answer": round(over, 2)})


# ---------- T6: outstanding principal after k annuity payments ----------- #
def t_loan_balance(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    p = float(rng.randint(10, 300) * 1_000_000)
    r = round(rng.uniform(10.0, 26.0), 2)
    years = rng.randint(2, 7)
    n = years * 12
    k = rng.randint(3, n - 3)
    i = r / 100 / 12
    a = _annuity(p, r, n)
    bal = p * (1 + i) ** k - a * ((1 + i) ** k - 1) / i
    q = {
        "en": (f"{COMPANIES_EN[co]} LLP takes an annuity loan of KZT {money(p,'en')} for {years} "
               f"years at {pct(r,'en')}% per year, repaid in equal monthly payments. Calculate "
               f"the outstanding principal after {k} monthly payments have been made."),
        "ru": (f"ТОО «{co}» взяло аннуитетный кредит на {money(p,'ru')} тенге сроком {years} "
               f"лет под {pct(r,'ru')}% годовых с ежемесячными равными платежами. Рассчитайте "
               f"остаток основного долга после внесения {k} ежемесячных платежей."),
        "kk": (f"«{co}» ЖШС {money(p,'kk')} теңгеге {years} жылға жылдық {pct(r,'kk')}% "
               f"мөлшерлемемен аннуитеттік несие алды, төлемдер ай сайын тең сомамен "
               f"жүргізіледі. {k} ай сайынғы төлем жасалғаннан кейінгі негізгі борыштың "
               f"қалдығын есептеңіз."),
        "kk_cs": (f"«{co}» ТОО {money(p,'kk')} теңгеге {years} жылға аннуитетный кредит алды, "
                  f"ставка жылына {pct(r,'kk')}%, платеж ай сайын бірдей. {k} платеж "
                  f"төленгеннен кейін остаток основного долга қанша қалады?"),
    }
    steps = [f"i = {i:.6f}, A = {a:.2f}",
             f"B_k = P(1+i)^k − A((1+i)^k − 1)/i, k = {k} → {bal:.2f}"]
    return _finish({"template": "loan_balance", "level": "advanced",
                    "question": q, "steps": steps, "answer": round(bal, 2)})


# ---------- T7: contractual penalty for late payment, with a cap -------- #
def t_contract_penalty(rng: random.Random) -> dict:
    import datetime as dt
    co = rng.choice(COMPANIES)
    debt = float(rng.randint(2, 200) * 500_000)
    rate = rng.choice([0.05, 0.1, 0.15, 0.2, 0.3])        # % per day
    cap = rng.choice([10, 15, 20])                         # % of debt
    due = dt.date(2025, 1, 1) + dt.timedelta(days=rng.randint(0, 540))
    days = rng.randint(10, 200)
    paid = due + dt.timedelta(days=days)
    raw = debt * rate / 100 * days
    pen = min(raw, debt * cap / 100)
    q = {
        "en": (f"Under a supply contract, {COMPANIES_EN[co]} LLP had to pay KZT {money(debt,'en')}; "
               f"the payment deadline was {date_str(due,'en')}. Payment was received on "
               f"{date_str(paid,'en')}. The contract sets a penalty of {pct(rate,'en')}% of the "
               f"unpaid amount for each day of delay, counted from the day after the deadline up to "
               f"and including the payment date, but not more than {cap}% of the unpaid amount. "
               f"Calculate the penalty."),
        "ru": (f"По договору поставки ТОО «{co}» должно было оплатить {money(debt,'ru')} тенге, "
               f"срок оплаты — {date_str(due,'ru')}. Оплата поступила {date_str(paid,'ru')}. "
               f"Договором предусмотрена неустойка {pct(rate,'ru')}% от неоплаченной суммы за "
               f"каждый день просрочки, начиная со дня, следующего за сроком оплаты, по день "
               f"оплаты включительно, но не более {cap}% от неоплаченной суммы. Рассчитайте "
               f"сумму неустойки."),
        "kk": (f"Жеткізу шарты бойынша «{co}» ЖШС {money(debt,'kk')} теңге төлеуге тиіс еді, "
               f"төлем мерзімі — {date_str(due,'kk')}. Төлем {date_str(paid,'kk')} күні түсті. "
               f"Шартта мерзімі өткен әрбір күн үшін төленбеген соманың {pct(rate,'kk')}% "
               f"мөлшерінде тұрақсыздық айыбы көзделген: ол төлем мерзімінен кейінгі күннен "
               f"бастап төлем күнін қоса алғанда есептеледі, бірақ төленбеген соманың "
               f"{cap}%-ынан аспайды. Тұрақсыздық айыбының сомасын есептеңіз."),
        "kk_cs": (f"Поставка договоры бойынша «{co}» ТОО {money(debt,'kk')} теңге төлеу керек еді, "
                  f"срок оплаты — {date_str(due,'kk')}. Оплата {date_str(paid,'kk')} күні келді. "
                  f"Договорда неустойка бар: просрочканың әр күніне неоплаченная сумманың "
                  f"{pct(rate,'kk')}%-ы, срок біткен келесі күннен бастап оплата күнін қоса, "
                  f"бірақ сумманың {cap}%-ынан көп емес. Неустойка қанша болады?"),
    }
    steps = [f"Days of delay = {date_str(paid,'en')} − {date_str(due,'en')} = {days}",
             f"Penalty before cap = {debt:.2f} × {rate}% × {days} = {raw:.2f}",
             f"Cap = {cap}% × {debt:.2f} = {debt*cap/100:.2f}; penalty = {pen:.2f}"]
    return _finish({"template": "contract_penalty", "level": "intermediate",
                    "question": q, "steps": steps, "answer": round(pen, 2)})


# ---------- T8: VAT included in a gross amount -------------------------- #
def t_vat_extraction(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    v = rng.choice([12, 16])
    gross = float(rng.randint(100, 50_000) * 1_000)
    vat = gross * v / (100 + v)
    q = {
        "en": (f"An invoice issued by {COMPANIES_EN[co]} LLP totals KZT {money(gross,'en')}, "
               f"including VAT at {v}%. Calculate the amount of VAT included in the invoice."),
        "ru": (f"Сумма счёта-фактуры ТОО «{co}» составляет {money(gross,'ru')} тенге, включая "
               f"НДС по ставке {v}%. Рассчитайте сумму НДС в составе счёта."),
        "kk": (f"«{co}» ЖШС шот-фактурасының сомасы {v}% мөлшерлемесі бойынша ҚҚС-ты қоса "
               f"алғанда {money(gross,'kk')} теңге. Шоттың құрамындағы ҚҚС сомасын есептеңіз."),
        "kk_cs": (f"«{co}» ТОО счет-фактурасы {money(gross,'kk')} теңге, НДС {v}% ішінде. "
                  f"Осы сумманың ішіндегі НДС қанша, выделить етіп беріңізші."),
    }
    steps = [f"VAT = G × v / (100 + v) = {gross:.2f} × {v}/{100+v} = {vat:.2f}"]
    return _finish({"template": "vat_extraction", "level": "basic",
                    "question": q, "steps": steps, "answer": round(vat, 2)})


# ---------- T9: straight-line depreciation, carrying amount ------------- #
def t_depreciation(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    cost = float(rng.randint(5, 500) * 1_000_000)
    life = rng.randint(4, 15)
    salv = float(round(cost * rng.uniform(0.0, 0.15) / 100_000) * 100_000)
    k = rng.randint(1, life - 1)
    carry = cost - (cost - salv) / life * k
    q = {
        "en": (f"{COMPANIES_EN[co]} LLP bought equipment for KZT {money(cost,'en')}. Its useful life "
               f"is {life} years and its residual value is KZT {money(salv,'en')}. Using the "
               f"straight-line method, calculate the carrying amount after {k} full years."),
        "ru": (f"ТОО «{co}» приобрело оборудование за {money(cost,'ru')} тенге. Срок полезного "
               f"использования — {life} лет, ликвидационная стоимость — {money(salv,'ru')} тенге. "
               f"Рассчитайте балансовую стоимость оборудования через {k} полных лет при "
               f"прямолинейном методе амортизации."),
        "kk": (f"«{co}» ЖШС жабдықты {money(cost,'kk')} теңгеге сатып алды. Пайдалы қызмет "
               f"мерзімі — {life} жыл, тарату құны — {money(salv,'kk')} теңге. Біркелкі (тура "
               f"сызықты) амортизация әдісімен {k} толық жылдан кейінгі жабдықтың баланстық "
               f"құнын есептеңіз."),
        "kk_cs": (f"«{co}» ТОО оборудование алды {money(cost,'kk')} теңгеге. Срок полезного "
                  f"использования {life} жыл, ликвидационная стоимость {money(salv,'kk')} теңге. "
                  f"Прямолинейный методпен амортизация есептесек, {k} жылдан кейін балансовая "
                  f"стоимость қанша болады?"),
    }
    steps = [f"Annual depreciation = ({cost:.0f} − {salv:.0f}) / {life} = {(cost-salv)/life:.2f}",
             f"Carrying amount = {cost:.0f} − {(cost-salv)/life:.2f} × {k} = {carry:.2f}"]
    return _finish({"template": "depreciation", "level": "basic",
                    "question": q, "steps": steps, "answer": round(carry, 2)})


# ---------- T10: return on average equity ------------------------------ #
def t_roe(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    e0 = float(rng.randint(100, 5000) * 1_000_000)
    e1 = float(round(e0 * rng.uniform(0.85, 1.35) / 1_000_000) * 1_000_000)
    ni = float(round((e0 + e1) / 2 * rng.uniform(0.02, 0.30) / 100_000) * 100_000)
    roe = ni / ((e0 + e1) / 2) * 100
    q = {
        "en": (f"{COMPANIES_EN[co]} LLP reported net profit of KZT {money(ni,'en')} for the year. "
               f"Equity was KZT {money(e0,'en')} at the beginning of the year and KZT "
               f"{money(e1,'en')} at the end. Calculate return on equity (ROE) based on average "
               f"equity, in percent."),
        "ru": (f"Чистая прибыль ТОО «{co}» за год составила {money(ni,'ru')} тенге. Собственный "
               f"капитал на начало года — {money(e0,'ru')} тенге, на конец года — "
               f"{money(e1,'ru')} тенге. Рассчитайте рентабельность собственного капитала (ROE) "
               f"по среднему значению капитала, в процентах."),
        "kk": (f"«{co}» ЖШС-нің жылдық таза пайдасы {money(ni,'kk')} теңге болды. Меншікті "
               f"капитал жыл басында — {money(e0,'kk')} теңге, жыл соңында — {money(e1,'kk')} "
               f"теңге. Капиталдың орташа шамасы бойынша меншікті капиталдың рентабельділігін "
               f"(ROE) пайызбен есептеңіз."),
        "kk_cs": (f"«{co}» ТОО-ның чистая прибылі жылына {money(ni,'kk')} теңге. Собственный "
                  f"капитал год басында {money(e0,'kk')} теңге, год соңында {money(e1,'kk')} "
                  f"теңге. Средний капиталмен ROE қанша процент шығады?"),
    }
    steps = [f"Average equity = ({e0:.0f} + {e1:.0f}) / 2 = {(e0+e1)/2:.2f}",
             f"ROE = {ni:.0f} / {(e0+e1)/2:.2f} × 100 = {roe:.4f}%"]
    return _finish({"template": "roe", "level": "intermediate",
                    "question": q, "steps": steps, "answer": round(roe, 2)})


# ---------- T11: net present value of a project ------------------------- #
def t_npv(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    while True:
        inv = float(rng.randint(50, 2000) * 1_000_000)
        n = rng.randint(3, 5)
        cfs = [float(round(inv * rng.uniform(0.15, 0.55) / 1_000_000) * 1_000_000) for _ in range(n)]
        d = rng.choice([8, 10, 12, 14, 15, 16, 18, 20])
        npv = -inv + sum(cf / (1 + d / 100) ** (t + 1) for t, cf in enumerate(cfs))
        if abs(npv) >= 0.05 * inv:          # keep the answer away from zero
            break

    def flows(lang: str) -> str:
        if lang == "en":
            return "; ".join(f"year {t+1} — KZT {money(cf,'en')}" for t, cf in enumerate(cfs))
        if lang == "ru":
            return "; ".join(f"{t+1}-й год — {money(cf,'ru')} тенге" for t, cf in enumerate(cfs))
        return "; ".join(f"{t+1}-жыл — {money(cf,'kk')} теңге" for t, cf in enumerate(cfs))

    q = {
        "en": (f"{COMPANIES_EN[co]} LLP is considering a project with an initial investment of KZT "
               f"{money(inv,'en')}. Expected net cash flows: {flows('en')}. The discount rate is "
               f"{d}%. Calculate the project's net present value (NPV)."),
        "ru": (f"ТОО «{co}» рассматривает проект с первоначальными инвестициями {money(inv,'ru')} "
               f"тенге. Ожидаемые чистые денежные потоки: {flows('ru')}. Ставка дисконтирования — "
               f"{d}%. Рассчитайте чистую приведённую стоимость проекта (NPV)."),
        "kk": (f"«{co}» ЖШС бастапқы инвестициясы {money(inv,'kk')} теңге болатын жобаны "
               f"қарастырып отыр. Күтілетін таза ақша ағындары: {flows('kk')}. Дисконттау "
               f"мөлшерлемесі — {d}%. Жобаның таза келтірілген құнын (NPV) есептеңіз."),
        "kk_cs": (f"«{co}» ТОО бір проектті қарап жатыр, инвестиция {money(inv,'kk')} теңге. "
                  f"Денежный потоктар: {flows('kk')}. Ставка дисконтирования {d}%. NPV қанша "
                  f"шығады?"),
    }
    steps = [f"PV(CF_t) = CF_t / (1 + {d/100})^t",
             f"NPV = −{inv:.0f} + Σ PV = {npv:.2f}"]
    return _finish({"template": "npv", "level": "intermediate",
                    "question": q, "steps": steps, "answer": round(npv, 2)})


# ---------- T12: FX loss on revaluation of a USD loan ------------------- #
def t_fx_revaluation(rng: random.Random) -> dict:
    co = rng.choice(COMPANIES)
    usd = float(rng.randint(10, 2000) * 10_000)
    r0 = round(rng.uniform(450.0, 550.0), 2)
    r1 = round(r0 + rng.uniform(3.0, 40.0), 2)
    loss = usd * (r1 - r0)
    q = {
        "en": (f"{COMPANIES_EN[co]} LLP has a bank loan of USD {money(usd,'en')}. The exchange rate "
               f"was KZT {pct(r0,'en')} per USD at the beginning of the quarter and KZT "
               f"{pct(r1,'en')} per USD at the end. Calculate the foreign exchange loss on "
               f"revaluation of the loan, in tenge."),
        "ru": (f"У ТОО «{co}» есть банковский кредит в размере {money(usd,'ru')} долларов США. "
               f"Курс на начало квартала — {pct(r0,'ru')} тенге за доллар, на конец квартала — "
               f"{pct(r1,'ru')} тенге за доллар. Рассчитайте курсовую разницу (убыток) от "
               f"переоценки кредита в тенге."),
        "kk": (f"«{co}» ЖШС-нің {money(usd,'kk')} АҚШ доллары мөлшеріндегі банк несиесі бар. "
               f"Тоқсан басындағы бағам — 1 доллар үшін {pct(r0,'kk')} теңге, тоқсан "
               f"соңындағы бағам — {pct(r1,'kk')} теңге. Несиені қайта бағалаудан туындайтын "
               f"бағамдық айырманы (шығынды) теңгемен есептеңіз."),
        "kk_cs": (f"«{co}» ТОО-да {money(usd,'kk')} долларлық кредит бар. Квартал басында курс "
                  f"{pct(r0,'kk')} теңге болған, квартал соңында {pct(r1,'kk')} теңге. "
                  f"Переоценкадан курсовая разница қанша, убыток теңгемен?"),
    }
    steps = [f"FX loss = {usd:.2f} × ({r1} − {r0}) = {loss:.2f}"]
    return _finish({"template": "fx_revaluation", "level": "basic",
                    "question": q, "steps": steps, "answer": round(loss, 2)})


TEMPLATES = [t_simple_interest, t_annuity_payment, t_current_ratio, t_deposit_compound,
             t_loan_overpayment, t_loan_balance, t_contract_penalty, t_vat_extraction,
             t_depreciation, t_roe, t_npv, t_fx_revaluation]
REGISTERS = ["en", "ru", "kk", "kk_kb", "kk_cs", "kk_cs_kb"]

In [ ]:
%%writefile generate.py
"""Generate the pilot set: every instance in every register, one JSONL row each.

    python generate.py --per_template 20 --out data/pilot.jsonl
"""
import argparse
import json
import pathlib
import random

from templates_kk import REGISTERS, TEMPLATES


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--per_template", type=int, default=20)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--out", default="data/pilot.jsonl")
    args = ap.parse_args()

    master = random.Random(args.seed)
    out = pathlib.Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with out.open("w", encoding="utf-8") as f:
        for tpl in TEMPLATES:
            for k in range(args.per_template):
                seed = master.randint(1, 2**31 - 1)
                item = tpl(random.Random(seed))
                uid = f"{item['template']}-{k:03d}"
                for reg in REGISTERS:
                    f.write(json.dumps({
                        "uid": uid, "register": reg, "template": item["template"],
                        "level": item["level"], "seed": seed,
                        "question": item["question"][reg],
                        "answer": item["answer"], "steps": item["steps"],
                    }, ensure_ascii=False) + "\n")
                    n += 1
    print(f"wrote {n} rows ({n // len(REGISTERS)} instances x {len(REGISTERS)} registers) -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile score.py
"""Score predictions: final-answer accuracy per register + paired tests vs 'kk'.

    python score.py --data data/pilot.jsonl --preds preds/sherkala.jsonl

A prediction is correct if its final number is within 1% (relative) of the
gold answer. Because every instance exists in every register with the same
gold answer, registers are compared pairwise with an exact McNemar test.

Note: FinChain's ChainEval step alignment uses English-only embedders, so for
Kazakh we report final-answer accuracy; step-level scoring needs a
multilingual embedder (e.g. LaBSE or multilingual-e5) - a later step.
"""
import argparse
import json
import re
from collections import defaultdict
from math import comb

ANSWER_MARK = re.compile(r"(?:answer|ответ|жауабы|жауап)\s*[:：]?", re.I)
NUM = re.compile(r"-?\d[\d\s  ,.]*")


def parse_number(text: str):
    """Take the number after the last answer marker, else the last number."""
    m = list(ANSWER_MARK.finditer(text))
    tail = text[m[-1].end():] if m else text
    cands = NUM.findall(tail) if m else NUM.findall(text)
    if not cands:
        return None
    raw = (cands[0] if m else cands[-1]).strip().rstrip(".,")
    raw = re.sub(r"[\s  ]", "", raw)
    if "," in raw and "." in raw:            # 1,250,000.50  or 1.250.000,50
        raw = raw.replace(",", "") if raw.rfind(".") > raw.rfind(",") else \
            raw.replace(".", "").replace(",", ".")
    elif "," in raw:                         # 1250000,50 (decimal) or 1,250,000
        parts = raw.split(",")
        raw = raw.replace(",", ".") if len(parts) == 2 and len(parts[1]) != 3 \
            else raw.replace(",", "")
    try:
        return float(raw)
    except ValueError:
        return None


def is_correct(pred, gold, rel_tol=0.01) -> bool:
    if pred is None:
        return False
    return abs(pred - gold) <= rel_tol * max(abs(gold), 1e-9)


def mcnemar_exact(b: int, c: int) -> float:
    """Two-sided exact McNemar p-value from discordant counts b, c."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    p = 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, p)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/pilot.jsonl")
    ap.add_argument("--preds", required=True)
    ap.add_argument("--baseline", default="kk")
    args = ap.parse_args()

    gold = {(r["uid"], r["register"]): r["answer"]
            for r in map(json.loads, open(args.data, encoding="utf-8"))}
    ok = {}
    for p in map(json.loads, open(args.preds, encoding="utf-8")):
        key = (p["uid"], p["register"])
        ok[key] = is_correct(parse_number(p["output"]), gold[key])

    by_reg = defaultdict(list)
    for (uid, reg), v in ok.items():
        by_reg[reg].append(v)
    uids = sorted({u for u, _ in ok})
    print(f"{'register':10} {'acc':>6} {'n':>5}   vs {args.baseline}: b/c   p(McNemar)")
    for reg in ["en", "ru", "kk", "kk_kb", "kk_cs", "kk_cs_kb"]:
        if reg not in by_reg:
            continue
        acc = sum(by_reg[reg]) / len(by_reg[reg])
        line = f"{reg:10} {acc:6.1%} {len(by_reg[reg]):5}"
        if reg != args.baseline:
            b = sum(ok.get((u, args.baseline), False) and not ok.get((u, reg), False) for u in uids)
            c = sum(not ok.get((u, args.baseline), False) and ok.get((u, reg), False) for u in uids)
            line += f"   {b:>3}/{c:<3}   {mcnemar_exact(b, c):.3f}"
        print(line)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile hf_runner.py
"""Batched greedy generation with Hugging Face transformers.

Written for Kaggle "GPU T4 x2" (2 x 15 GB): an 8B model in fp16 is split
across both cards with device_map="auto". Works on any CUDA GPU, and on CPU
for tiny smoke-test models.
"""
from __future__ import annotations

import gc

# Fallback for models whose tokenizer ships without a chat template
# (Llama-3 style, which Sherkala and KazLLM are built on).
LLAMA3_TEMPLATE = (
    "{{ bos_token }}{% for m in messages %}"
    "<|start_header_id|>{{ m['role'] }}<|end_header_id|>\n\n{{ m['content'] }}<|eot_id|>"
    "{% endfor %}{% if add_generation_prompt %}"
    "<|start_header_id|>assistant<|end_header_id|>\n\n{% endif %}"
)


def load(model_id: str, dtype: str = "float16"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.chat_template is None:
        tok.chat_template = LLAMA3_TEMPLATE
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    use_cuda = torch.cuda.is_available()
    kw = {"device_map": "auto" if use_cuda else None}
    dt = getattr(torch, dtype) if use_cuda else torch.float32
    try:                                   # transformers >= 4.56 uses `dtype`
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dt, **kw)
    except TypeError:                      # older releases: `torch_dtype`
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dt, **kw)
    model.eval()
    return tok, model


def _stop_ids(tok, model) -> list[int]:
    ids = set()
    for src in (model.generation_config.eos_token_id, tok.eos_token_id):
        if isinstance(src, int):
            ids.add(src)
        elif src:
            ids.update(src)
    for t in ("<|eot_id|>", "<|im_end|>", "<|end_of_text|>"):
        tid = tok.convert_tokens_to_ids(t)
        if isinstance(tid, int) and tid != tok.unk_token_id and tid is not None:
            ids.add(tid)
    return sorted(ids)


def generate(tok, model, conversations: list[list[dict]], max_new_tokens: int = 768,
             batch_size: int = 16, desc: str = "") -> list[str]:
    """Greedy-decode every conversation; returns outputs in input order."""
    import torch

    prompts = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True)
               for c in conversations]
    # Long prompts first: similar lengths share a batch, OOM shows up early.
    order = sorted(range(len(prompts)), key=lambda i: -len(prompts[i]))
    stop = _stop_ids(tok, model)
    device = next(model.parameters()).device
    outs: list[str | None] = [None] * len(prompts)
    try:
        from tqdm.auto import tqdm
        batches = tqdm(range(0, len(order), batch_size), desc=desc or "generate")
    except ImportError:
        batches = range(0, len(order), batch_size)
    for b in batches:
        idx = order[b:b + batch_size]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 eos_token_id=stop, pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        for j, i in enumerate(idx):
            outs[i] = tok.decode(new[j], skip_special_tokens=True).strip()
    return outs  # type: ignore[return-value]


def unload(model) -> None:
    import torch
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
%%writefile run_model.py
"""Run one model on the pilot set and save its answers.

    python run_model.py --model inceptionai/Llama-3.1-Sherkala-8B-Chat \
        --data data/pilot.jsonl --out preds/sherkala.jsonl

Gated models (Sherkala, KazLLM) need an accepted licence on Hugging Face and
HF_TOKEN in the environment.
"""
import argparse
import json
import pathlib

import hf_runner

# One instruction per prompt language. The instruction itself is NOT
# perturbed: only the task text differs between registers.
INSTRUCTION = {
    "en": "Solve the problem step by step. End with a line 'Answer: <number>'.",
    "ru": "Решите задачу пошагово. В конце напишите строку 'Ответ: <число>'.",
    "kk": "Есепті қадам бойынша шығарыңыз. Соңында 'Жауабы: <сан>' деп жазыңыз.",
}


def prompt_lang(register: str) -> str:
    return "en" if register == "en" else "ru" if register == "ru" else "kk"


def build_messages(row: dict) -> list:
    return [{"role": "user",
             "content": f"{row['question']}\n\n{INSTRUCTION[prompt_lang(row['register'])]}"}]


def run(model_id: str, rows: list[dict], out_path: str, max_new_tokens: int = 768,
        batch_size: int = 16, tok_model=None) -> None:
    tok, model = tok_model or hf_runner.load(model_id)
    outs = hf_runner.generate(tok, model, [build_messages(r) for r in rows],
                              max_new_tokens=max_new_tokens, batch_size=batch_size,
                              desc=model_id.split("/")[-1])
    pathlib.Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for r, o in zip(rows, outs):
            f.write(json.dumps({"uid": r["uid"], "register": r["register"],
                                "model": model_id, "output": o}, ensure_ascii=False) + "\n")
    if tok_model is None:
        hf_runner.unload(model)
    print(f"saved {len(outs)} answers -> {out_path}")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--data", default="data/pilot.jsonl")
    ap.add_argument("--out", required=True)
    ap.add_argument("--max_new_tokens", type=int, default=768)
    ap.add_argument("--batch_size", type=int, default=16)
    args = ap.parse_args()
    rows = [json.loads(l) for l in open(args.data, encoding="utf-8")]
    run(args.model, rows, args.out, args.max_new_tokens, args.batch_size)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile kazmmlu_flatten.py
"""Arm B: does Russian-keyboard spelling hurt models on KazMMLU (Kazakh part)?

Every Kazakh-language KazMMLU question is asked twice: as published and with
its question and options typed on a Russian keyboard (kk_surface). The
instruction line stays standard. Paired accuracy difference = cost of
keyboard spelling on someone else's benchmark.

    python kazmmlu_flatten.py --model inceptionai/Llama-3.1-Sherkala-8B-Chat --limit 100

KazMMLU: huggingface.co/datasets/MBZUAI/KazMMLU (CC BY-NC 4.0, research use).
Subjects are discovered from the repository ("... in kaz)" folders); column
names are detected automatically and printed on the first run.
"""
from __future__ import annotations

import argparse
import json
import pathlib
import re

import hf_runner
from kk_surface import keyboard_flatten

REPO = "MBZUAI/KazMMLU"
LETTERS = "ABCDE"
INSTR = "Тек дұрыс жауаптың әрпін жазыңыз (A, B, C, D немесе E)."


def kazakh_files() -> list[str]:
    from huggingface_hub import list_repo_files
    files = list_repo_files(REPO, repo_type="dataset")
    return sorted(f for f in files if "in kaz)" in f and f.endswith("test.csv"))


def _columns(df):
    q = next(c for c in df.columns if "question" in c.lower())
    opts = [c for c in df.columns if re.search(r"option", c, re.I)]
    ans = next(c for c in df.columns if "answer" in c.lower())
    return q, opts, ans


def _prompt(question: str, options: list, flatten: bool) -> str:
    f = keyboard_flatten if flatten else (lambda s: s)
    lines = [f(str(question).strip())]
    for letter, opt in zip(LETTERS, options):
        if isinstance(opt, str) and opt.strip():
            lines.append(f"{letter}. {f(opt.strip())}")
    lines.append(INSTR)
    return "\n".join(lines)


def build_items(limit: int = 100) -> list[dict]:
    import pandas as pd
    from huggingface_hub import hf_hub_download
    items, shown = [], False
    for path in kazakh_files():
        df = pd.read_csv(hf_hub_download(REPO, path, repo_type="dataset"))
        q, opts, ans = _columns(df)
        if not shown:
            print("KazMMLU columns:", list(df.columns))
            shown = True
        subject = path.split("/")[0]
        for k, row in df.head(limit).iterrows():
            gold = str(row[ans]).strip()[:1].upper()
            if gold not in LETTERS:
                continue
            options = [row[c] for c in opts]
            items.append({"uid": f"{subject}#{k}", "subject": subject, "gold": gold,
                          "std": _prompt(row[q], options, False),
                          "kb": _prompt(row[q], options, True)})
    print(f"KazMMLU: {len(items)} Kazakh questions from {len(kazakh_files())} subjects")
    return items


def _letter(text: str) -> str:
    m = re.search(r"\b([A-E])\b", text.upper()) or re.search(r"([A-E])", text.upper())
    return m.group(1) if m else "?"


def run(model_id: str, items: list[dict], out_path: str, batch_size: int = 32,
        tok_model=None) -> dict:
    tok, model = tok_model or hf_runner.load(model_id)
    res = {}
    for form in ("std", "kb"):
        outs = hf_runner.generate(tok, model, [[{"role": "user", "content": it[form]}] for it in items],
                                  max_new_tokens=8, batch_size=batch_size,
                                  desc=f"KazMMLU {form}")
        res[form] = [_letter(o) for o in outs]
    if tok_model is None:
        hf_runner.unload(model)
    pathlib.Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for it, s, k in zip(items, res["std"], res["kb"]):
            f.write(json.dumps({"uid": it["uid"], "subject": it["subject"], "gold": it["gold"],
                                "pred_std": s, "pred_kb": k, "model": model_id},
                               ensure_ascii=False) + "\n")
    return summarize(out_path)


def summarize(path: str) -> dict:
    from score import mcnemar_exact
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    n = len(rows)
    ok_s = [r["pred_std"] == r["gold"] for r in rows]
    ok_k = [r["pred_kb"] == r["gold"] for r in rows]
    b = sum(s and not k for s, k in zip(ok_s, ok_k))
    c = sum(k and not s for s, k in zip(ok_s, ok_k))
    out = {"model": rows[0]["model"] if rows else "", "n": n,
           "acc_std": sum(ok_s) / max(n, 1), "acc_kb": sum(ok_k) / max(n, 1),
           "lost": b, "gained": c, "p_mcnemar": mcnemar_exact(b, c)}
    print(f"{out['model']}: n={n} standard={out['acc_std']:.1%} keyboard={out['acc_kb']:.1%} "
          f"lost={b} gained={c} p={out['p_mcnemar']:.4f}")
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--limit", type=int, default=100, help="questions per subject")
    ap.add_argument("--out", default=None)
    args = ap.parse_args()
    out = args.out or f"preds/kazmmlu_{args.model.split('/')[-1]}.jsonl"
    run(args.model, build_items(args.limit), out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile report.py
"""Summarise all model runs: tables + one chart.

    python report.py --data data/pilot.jsonl --preds preds --out results

Writes
  results/summary.csv      accuracy per model x register, drop vs kk, McNemar p
  results/by_template.csv  accuracy per model x template x register
  results/accuracy.png     grouped bars: accuracy by register, one colour per model
"""
import argparse
import csv
import glob
import json
import pathlib
from collections import defaultdict

from score import is_correct, mcnemar_exact, parse_number

REGISTERS = ["en", "ru", "kk", "kk_kb", "kk_cs", "kk_cs_kb"]
LABELS = {"en": "English", "ru": "Russian", "kk": "Kazakh\n(standard)",
          "kk_kb": "Kazakh\n(RU keyboard)", "kk_cs": "Kazakh–Russian\nmix",
          "kk_cs_kb": "Mix +\nRU keyboard"}
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]  # fixed order


def load_scores(data_path: str, pred_files: list[str]):
    gold = {}
    meta = {}
    for r in map(json.loads, open(data_path, encoding="utf-8")):
        gold[(r["uid"], r["register"])] = r["answer"]
        meta[r["uid"]] = r["template"]
    ok = defaultdict(dict)                      # model -> (uid, reg) -> bool
    for path in pred_files:
        for p in map(json.loads, open(path, encoding="utf-8")):
            key = (p["uid"], p["register"])
            if key in gold:
                ok[p["model"]][key] = is_correct(parse_number(p["output"]), gold[key])
    return ok, meta


def summarise(ok, meta, out_dir: pathlib.Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    rows, trows = [], []
    for model, res in ok.items():
        uids = sorted({u for u, _ in res})
        base = {u: res.get((u, "kk"), False) for u in uids}
        for reg in REGISTERS:
            vals = [res[(u, reg)] for u in uids if (u, reg) in res]
            if not vals:
                continue
            acc = sum(vals) / len(vals)
            b = sum(base[u] and not res.get((u, reg), False) for u in uids)
            c = sum(not base[u] and res.get((u, reg), False) for u in uids)
            rows.append({"model": model.split("/")[-1], "register": reg, "n": len(vals),
                         "accuracy": round(acc, 4),
                         "drop_vs_kk_pp": "" if reg == "kk" else round(
                             100 * (sum(base.values()) / len(uids) - acc), 1),
                         "lost_vs_kk": "" if reg == "kk" else b,
                         "gained_vs_kk": "" if reg == "kk" else c,
                         "p_mcnemar": "" if reg == "kk" else round(mcnemar_exact(b, c), 4)})
        by_t = defaultdict(list)
        for (u, reg), v in res.items():
            by_t[(meta[u], reg)].append(v)
        for (t, reg), vals in sorted(by_t.items()):
            trows.append({"model": model.split("/")[-1], "template": t, "register": reg,
                          "n": len(vals), "accuracy": round(sum(vals) / len(vals), 4)})
    for name, data in (("summary.csv", rows), ("by_template.csv", trows)):
        with open(out_dir / name, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(data[0]))
            w.writeheader()
            w.writerows(data)
    return rows


def chart(rows, out_png: pathlib.Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    models = list(dict.fromkeys(r["model"] for r in rows))
    acc = {(r["model"], r["register"]): r["accuracy"] for r in rows}
    regs = [g for g in REGISTERS if any((m, g) in acc for m in models)]
    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=200)
    width = 0.8 / len(models)
    for k, m in enumerate(models):
        xs = [i - 0.4 + width * (k + 0.5) for i in range(len(regs))]
        ys = [100 * acc.get((m, g), 0) for g in regs]
        ax.bar(xs, ys, width=width * 0.92, color=SERIES[k % len(SERIES)], label=m,
               edgecolor="white", linewidth=1)
    ax.set_xticks(range(len(regs)), [LABELS[g] for g in regs], fontsize=9, color="#3d3d3a")
    ax.set_ylabel("Final-answer accuracy, %", color="#3d3d3a")
    ax.set_ylim(0, 100)
    ax.yaxis.grid(True, color="#e8e7e2", linewidth=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#b9b8b1")
    ax.tick_params(axis="y", colors="#6f6e69", length=0)
    ax.tick_params(axis="x", length=0)
    ax.set_title("Same financial tasks, different surface forms", loc="left",
                 fontsize=12, color="#1f1f1d")
    ax.legend(frameon=False, ncol=len(models), loc="upper right", fontsize=9)
    fig.tight_layout()
    fig.savefig(out_png)
    print(f"chart -> {out_png}")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/pilot.jsonl")
    ap.add_argument("--preds", default="preds")
    ap.add_argument("--out", default="results")
    args = ap.parse_args()
    files = sorted(f for f in glob.glob(f"{args.preds}/*.jsonl")
                   if not pathlib.Path(f).name.startswith("kazmmlu_"))
    ok, meta = load_scores(args.data, files)
    rows = summarise(ok, meta, pathlib.Path(args.out))
    for r in rows:
        print(r)
    chart(rows, pathlib.Path(args.out) / "accuracy.png")


if __name__ == "__main__":
    main()

In [ ]:
import importlib, json, random, pathlib
for mod in ["kk_surface", "templates_kk", "score", "hf_runner", "run_model", "kazmmlu_flatten", "report"]:
    if mod in sys.modules: importlib.reload(sys.modules[mod])
import templates_kk, run_model, kazmmlu_flatten, report, hf_runner

per_t = 2 if QUICK_TEST else PER_TEMPLATE
subprocess.run([sys.executable, "generate.py", "--per_template", str(per_t), "--out", "data/pilot.jsonl"], check=True)
rows = [json.loads(l) for l in open("data/pilot.jsonl", encoding="utf-8")]
print(len(rows), "задач;", "пример kk_cs_kb:", next(r for r in rows if r["register"] == "kk_cs_kb")["question"][:160])
kz_items = kazmmlu_flatten.build_items(10 if QUICK_TEST else KAZMMLU_LIMIT) if RUN_KAZMMLU else []

## Прогон моделей
Каждая модель загружается один раз; готовые файлы не пересчитываются, если прогон прервался.

In [ ]:
pathlib.Path("preds").mkdir(exist_ok=True)
kz_summaries = []
for m in MODELS:
    short = m.split("/")[-1]
    pilot_out, kz_out = f"preds/{short}.jsonl", f"preds/kazmmlu_{short}.jsonl"
    need_pilot = not pathlib.Path(pilot_out).exists()
    need_kz = RUN_KAZMMLU and not pathlib.Path(kz_out).exists()
    if not (need_pilot or need_kz):
        print("уже есть:", short); continue
    t0 = time.time()
    tm = hf_runner.load(m)
    if need_pilot:
        run_model.run(m, rows, pilot_out, MAX_NEW_TOKENS, BATCH_SIZE, tok_model=tm)
    if need_kz:
        kz_summaries.append(kazmmlu_flatten.run(m, kz_items, kz_out, batch_size=32, tok_model=tm))
    hf_runner.unload(tm[1]); del tm
    print(f"{short}: {(time.time() - t0) / 60:.1f} мин")

## Результаты

In [ ]:
subprocess.run([sys.executable, "report.py", "--data", "data/pilot.jsonl", "--preds", "preds", "--out", "results"], check=True)
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv("results/summary.csv"))
display(Image("results/accuracy.png"))
if RUN_KAZMMLU:
    kz = [kazmmlu_flatten.summarize(p) for p in sorted(map(str, pathlib.Path("preds").glob("kazmmlu_*.jsonl")))]
    pd.DataFrame(kz).to_csv("results/kazmmlu_summary.csv", index=False)
    display(pd.DataFrame(kz))

In [ ]:
import shutil
for d in ["data", "preds", "results"]:
    shutil.copytree(d, f"bundle/{d}", dirs_exist_ok=True)
shutil.make_archive("results", "zip", "bundle")
print("Готово: results.zip во вкладке Output →", os.path.getsize("results.zip") // 1024, "КБ")